In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
import joblib

In [28]:
df = pd.read_csv('/Users/viniciussouza/Codes/FIAP/FIAP_projeto_fase_1/src/data/books_data.csv')
df.drop(columns=["image_url", "id"], inplace=True)
df['price'] = df['price'].str.replace(r"[^\d.]", "", regex=True).astype(float)
df

,title,price,stock,category,rating
0,It's Only the Himalayas,45.17,In stock,Travel,2
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,In stock,Travel,4
2,See America: A Celebration of Our National Par...,48.87,In stock,Travel,3
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,In stock,Travel,2
4,Under the Tuscan Sun,37.33,In stock,Travel,3
...,...,...,...,...,...
995,Why the Right Went Wrong: Conservatism--From G...,52.65,In stock,Politics,4
996,Equal Is Unfair: America's Misguided Fight Aga...,56.86,In stock,Politics,1
997,Amid the Chaos,36.58,In stock,Cultural,1
998,Dark Notes,19.19,In stock,Erotica,5


In [32]:
X["stock"].unique()

array(['In stock'], dtype=object)

In [29]:
X = df[
    [
        "stock", 
        "category", 
        "rating"
    ]
]

y = df["price"]

In [33]:
# Converter Stock

X["stock"] = X["stock"].map({"In stock": 1, "Out of stock": 0})

/var/folders/h0/9cxt3flx6px_mz8j5myq3cjw0000gn/T/ipykernel_33522/1373648003.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["stock"] = X["stock"].map({"In stock": 1, "Out of stock": 0})


In [34]:
X

,stock,category,rating
0,1,Travel,2
1,1,Travel,4
2,1,Travel,3
3,1,Travel,2
4,1,Travel,3
...,...,...,...
995,1,Politics,4
996,1,Politics,1
997,1,Cultural,1
998,1,Erotica,5


In [38]:
# Pré-processamento (OneHotEncoder para categoria)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown="ignore"), ["category"])
    ],
    remainder="passthrough"
)

In [39]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

In [40]:
pipeline.fit(X, y)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [42]:
joblib.dump(pipeline, '/Users/viniciussouza/Codes/FIAP/FIAP_projeto_fase_1/src/ml/pkl_dev/book_price_model.pkl')

['/Users/viniciussouza/Codes/FIAP/FIAP_projeto_fase_1/src/ml/pkl_dev/book_price_model.pkl']

## Testando meu modelo

In [43]:
model = joblib.load('/Users/viniciussouza/Codes/FIAP/FIAP_projeto_fase_1/src/ml/pkl_dev/book_price_model.pkl')

In [58]:
# Novo livro para prever preço
new_book = pd.DataFrame([{
    "rating": 5,
    "stock": 1,  # In stock
    "category": "Classics"
}])

# Predição
pred = model.predict(new_book)[0]
print(f"Preço previsto: {pred:.2f}")

Preço previsto: 31.03
